# 10.1 国产 NPU 部署实践（仿真）

## 目的

覆盖昇腾 CANN、寒武纪 MagicMind、地平线 HBDK、瑞芯微 RKNN 的**共性流水线**，用仿真代码理解选型与适配要点（无需真实 SDK）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

## 国产 NPU 对比速查

In [ ]:
NPU_TABLE = [
    # name, tops, mem_gb, tdp_w, quant, model_range, sdk
    ("昇腾310P", 128, 16, 12, "INT8/INT4", "1.5B-7B", "CANN/AMCT"),
    ("寒武纪MLU370", 48, 16, 75, "INT8/INT16", "1.5B-7B", "MagicMind"),
    ("地平线J6", 34, 6, 15, "INT8", "0.5B-1.5B", "HBDK"),
    ("瑞芯微RK3588", 6, 8, 5, "INT8/FP16", "0.5B-1.5B", "RKNN"),
]
print(f"{'平台':14s} {'TOPS':>6} {'Mem':>5} {'TDP':>5} {'量化':12s} {'模型':10s} SDK")
for r in NPU_TABLE:
    print(f"{r[0]:14s} {r[1]:6d} {r[2]:5d} {r[3]:5d} {r[4]:12s} {r[5]:10s} {r[6]}")

## 统一部署流水线（抽象）

`PyTorch → ONNX/自定义IR → 厂商量化工具 → 图编译 → Runtime`

In [ ]:
from dataclasses import dataclass


@dataclass
class DeployJob:
    model_name: str
    params_b: float
    target: str
    quant: str


COMPAT = {
    "昇腾310P": {"formats": ["ONNX", "OM"], "quants": ["INT8", "INT4"], "max_b": 7},
    "寒武纪MLU370": {"formats": ["ONNX", "MM"], "quants": ["INT8", "INT16"], "max_b": 7},
    "地平线J6": {"formats": ["ONNX", "HBM"], "quants": ["INT8"], "max_b": 1.5},
    "瑞芯微RK3588": {"formats": ["ONNX", "RKNN"], "quants": ["INT8", "FP16"], "max_b": 1.5},
}


def plan_deploy(job: DeployJob) -> dict:
    spec = COMPAT[job.target]
    ok_size = job.params_b <= spec["max_b"] + 1e-9
    ok_q = job.quant in spec["quants"]
    steps = [
        "export ONNX (dynamic axes for seq)",
        f"quantize via {job.target} toolchain → {job.quant}",
        "compile graph / fuse ops",
        "validate layer cosine > 0.999",
        "package runtime + warmup",
    ]
    warnings = []
    if not ok_size:
        warnings.append("模型偏大：建议更强量化/蒸馏/换成更小 SLM")
    if not ok_q:
        warnings.append(f"量化 {job.quant} 可能不被支持，候选={spec['quants']}")
    if "DeepSeek" in job.model_name:
        warnings.append("MLA 需自定义 attention kernel 或回退 MHA")
    return {"ok": ok_size and ok_q, "steps": steps, "warnings": warnings, "formats": spec["formats"]}


for job in [
    DeployJob("Qwen2.5-1.5B", 1.5, "瑞芯微RK3588", "INT8"),
    DeployJob("Qwen2.5-7B", 7.0, "昇腾310P", "INT4"),
    DeployJob("DeepSeek-7B", 7.0, "地平线J6", "INT8"),
]:
    plan = plan_deploy(job)
    print(f"\n=== {job.model_name} → {job.target} ({job.quant}) ===")
    print("可部署:" , plan["ok"], "| 格式:", plan["formats"])
    for w in plan["warnings"]:
        print("[WARN]", w)
    for i, s in enumerate(plan["steps"], 1):
        print(f"  {i}. {s}")

## 国产开源模型适配注意点

In [ ]:
MODELS = [
    ("Qwen2.5/3", "标准 LLaMA-like", "最省心，GGUF/ONNX/QNN 均成熟"),
    ("DeepSeek", "MLA", "自定义 attention；无 kernel 则回退 MHA"),
    ("MiniCPM", "多模态", "ViT+LLM 分包部署，注意视觉 token 数"),
    ("ChatGLM", "特殊位置编码", "RoPE/位置编码需厂商算子支持"),
    ("Yi", "标准", "CPU/GGUF 路径顺畅"),
]
print("模型族适配提示:")
for m, arch, tip in MODELS:
    print(f"- {m:10s} [{arch}] {tip}")

## 小结

- 边缘服务器：昇腾 / 寒武纪；车载：地平线；IoT 低成本：RK3588。
- 先确认 **算子覆盖 + 量化位宽 + 内存**，再谈吞吐。
- MLA / 多模态是国产模型端侧适配的两大坑点。